# 03 — Strategy Subagents-as-Tools Tests

Validates the rewritten Strategist that uses **subagents as tools**
(pattern from `05_subagents_as_tool.ipynb`).

Architecture being tested:

```
Supervisor Strategist (create_deep_agent)  — Mongols vs Vietnamese
  └─ consult_civ_analyst      @tool  → CivilizationAnalyst sub-agent  → get_civilization_info
  └─ consult_counter_analyst  @tool  → CounterAnalyst sub-agent        → get_unit_counters
  └─ consult_unit_stats       @tool  → UnitStatsAdvisor sub-agent      → get_unit_stats
  └─ consult_building_advisor @tool  → BuildingAdvisor sub-agent       → get_building_info
  └─ consult_matchup_analyst  @tool  → MatchupAnalyst sub-agent        → get_civilization_info (both civs)
                                        ↑ seeded with player_civ + opponent_civ at build time
```

Tests:
1. Build and call each sub-agent tool directly (incl. matchup analyst)
2. Verify the Supervisor delegates to the right sub-agents
3. Full strategy invocation on synthetic snapshots (matchup-aware)
4. Streaming output from the Supervisor
5. Async tick + multi-civ test

In [1]:
import sys
sys.path.insert(0, "/mnt/e/Personal/Samarth/repository/AOE-Agent/src")

from aoe2_agent.schemas import GameStateSnapshot, DetectedUnit
from aoe2_agent.strategy.agent import (
    build_kb_subagent_tools,
    build_strategist,
    _snapshot_to_prompt,
    _extract_last_content,
)
print("Imports OK")

Imports OK


## 1 — Build the Four Sub-agent Tools

Each `@tool` internally runs its own `create_deep_agent`.

In [3]:
PLAYER_CIV   = "Mongols"
OPPONENT_CIV = "Vietnamese"

# build_kb_subagent_tools now requires both civs to seed the MatchupAnalyst
tools = build_kb_subagent_tools(PLAYER_CIV, OPPONENT_CIV)

# Unpack all five tools
(
    consult_strategic_reference,
    consult_civ_analyst,
    consult_counter_analyst,
    consult_unit_stats,
    consult_building_advisor,
    consult_matchup_analyst,
) = tools

print(f"Sub-agent tools built ({PLAYER_CIV} vs {OPPONENT_CIV}):")
for t in tools:
    print(f"  @tool  {t.name:30s}  — {t.description[:60]}…")

Sub-agent tools built (Mongols vs Vietnamese):
  @tool  consult_strategic_reference     — Consult the Strategic Reference Advisor for the AoE2 master …
  @tool  consult_matchup_analyst         — Consult the Matchup Analyst sub-agent for player vs opponent…
  @tool  consult_civ_analyst             — Consult the Civilization Analyst sub-agent for AoE2 civ bonu…
  @tool  consult_counter_analyst         — Consult the Counter Analyst sub-agent for unit counter relat…
  @tool  consult_unit_stats              — Consult the Unit Stats Advisor sub-agent for AoE2 unit HP, a…
  @tool  consult_building_advisor        — Consult the Building Advisor sub-agent for building costs, H…


## 2 — Test Each Sub-agent Tool Directly

In [6]:
#------- Consultant Analyst -------------------------------
print("====consult_strategic_reference=============")
result= consult_strategic_reference.invoke("what are the units archers weak with")

====consult_strategic_reference


In [7]:
result

'{\n  "units_archers_weak_with": ["spearman", "skirmisher"]\n}'

In [4]:
# ── Civilization Analyst ──────────────────────────────────────────────────
print("=== CivilizationAnalyst ===")
result = consult_civ_analyst.invoke("What are the Mongols' civilization bonuses?")
print(result)

=== CivilizationAnalyst ===
{
  "bonuses": [
    {
      "name": "Berserkergang",
      "bonus_type": "unique_unit_bonus",
      "bonus_value": "+20% speed"
    },
    {
      "name": "Mongol Conquests",
      "bonus_type": "villager_bonus",
      "bonus_value": "50 villagers available at the start of the game"
    }
  ],
  "help_text": "The Mongols are a unique civilization with bonuses to their units and villager availability."
}


In [ ]:
# Compare two civs
result = consult_civ_analyst.invoke(
    "Compare Britons and Franks — which civ has stronger cavalry?"
)
print(result)

In [ ]:
# ── Counter Analyst ───────────────────────────────────────────────────────
print("=== CounterAnalyst ===")
result = consult_counter_analyst.invoke("What units counter knights effectively?")
print(result)

In [ ]:
result = consult_counter_analyst.invoke(
    "My opponent is massing archers. What should I build to counter them?"
)
print(result)

In [ ]:
# ── Unit Stats Advisor ────────────────────────────────────────────────────
print("=== UnitStatsAdvisor ===")
result = consult_unit_stats.invoke("What are the HP, attack and cost of a Knight?")
print(result)

In [ ]:
result = consult_unit_stats.invoke(
    "Compare Archer vs Crossbowman: cost and attack. Which is more cost-efficient?"
)
print(result)

In [ ]:
# ── Building Advisor ──────────────────────────────────────────────────────
print("=== BuildingAdvisor ===")
result = consult_building_advisor.invoke("How much does a Castle cost and what does it do?")
print(result)

In [ ]:
result = consult_building_advisor.invoke(
    "I need to produce cavalry and archers. What buildings do I need to build?"
)
print(result)

### Matchup Analyst — player vs opponent civilization comparison

In [ ]:
# ── Matchup Analyst ───────────────────────────────────────────────────────
print(f"=== MatchupAnalyst: {PLAYER_CIV} vs {OPPONENT_CIV} ===")
result = consult_matchup_analyst.invoke(
    f"How does {PLAYER_CIV} compare against {OPPONENT_CIV}? "
    "What are my key strengths, the opponent's main threats, and the single most "
    "important thing I should do differently because of this matchup?"
)
print(result)

In [ ]:
# Early-game matchup query
result = consult_matchup_analyst.invoke(
    f"It's the Dark Age. I'm {PLAYER_CIV} and my opponent is {OPPONENT_CIV}. "
    "What should I watch out for from their early-game bonuses, "
    "and how should I adapt my opening?"
)
print(result)

## 3 — Build the Supervisor Strategist

In [ ]:
# build_strategist now accepts opponent_civ — matchup analyst is pre-seeded
supervisor = build_strategist(PLAYER_CIV, OPPONENT_CIV)

print(f"Supervisor Strategist built for: {PLAYER_CIV} vs {OPPONENT_CIV}")
print(type(supervisor))

## 4 — Strategy Invocation: Supervisor Delegates to Sub-agents

In [ ]:
from collections import deque

# ── Dark Age opening ──────────────────────────────────────────────────────
dark_snap = GameStateSnapshot(
    age="Castle",
    population=100,
    pop_cap=10,
    resources={"food": 400, "wood": 250, "gold": 1000, "stone": 500},
    visible_units=[DetectedUnit(label="villager", owner="self", approx_count=16),DetectedUnit(label='knight',owner='self',approx_count=10),DetectedUnit(label='Archers',owner='self',approx_count=10)],
    visible_buildings=["Town Center", "House"],
    idle_villagers_visible=False,
    threat_level="low",
    notes="Game has reached in build army phase",
)

prompt = _snapshot_to_prompt(dark_snap, deque(), PLAYER_CIV, OPPONENT_CIV)
print("--- Prompt ---")
print(prompt)

In [ ]:
# Invoke — watch the supervisor call sub-agent tools
result = supervisor.invoke({"messages": [{"role": "user", "content": prompt}]})

# Show the full message chain (includes tool calls and sub-agent responses)
print("=== Full message chain ===")
for i, msg in enumerate(result.get("messages", [])):
    role = getattr(msg, "type", type(msg).__name__)
    content = getattr(msg, "content", "") or ""
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        print(f"[{i}] {role}: calls → {[tc['name'] for tc in tool_calls]}")
    elif content:
        preview = str(content)[:120].replace("\n", " ")
        print(f"[{i}] {role}: {preview}")

In [ ]:
# Final strategy output
strategy = _extract_last_content(result)
print("=== Strategy Update ===")
print(strategy)

In [ ]:
# ── Castle Age under attack ───────────────────────────────────────────────
attack_snap = GameStateSnapshot(
    age="Castle",
    population=95,
    pop_cap=100,
    resources={"food": 150, "wood": 200, "gold": 80, "stone": 0},
    visible_units=[
        DetectedUnit(label="knight", owner="enemy", approx_count=8),
        DetectedUnit(label="villager", owner="self", approx_count=12),
        DetectedUnit(label="spearman", owner="self", approx_count=4),
    ],
    visible_buildings=["Town Center", "Castle", "Stable", "Barracks"],
    idle_villagers_visible=True,
    threat_level="high",
    notes="8 enemy knights approaching Town Center from the east",
)

prompt2 = _snapshot_to_prompt(attack_snap, deque(), PLAYER_CIV, OPPONENT_CIV)
result2 = supervisor.invoke({"messages": [{"role": "user", "content": prompt2}]})
print(_extract_last_content(result2))

In [ ]:
# ── Imperial Age siege push ───────────────────────────────────────────────
imperial_snap = GameStateSnapshot(
    age="Imperial",
    population=180,
    pop_cap=200,
    resources={"food": 1200, "wood": 800, "gold": 500, "stone": 300},
    visible_units=[
        DetectedUnit(label="trebuchet", owner="self", approx_count=3),
        DetectedUnit(label="cavalry archer", owner="self", approx_count=20),
        DetectedUnit(label="halberdier", owner="enemy", approx_count=10),
    ],
    visible_buildings=["Town Center", "Castle", "Siege Workshop", "University"],
    idle_villagers_visible=False,
    threat_level="low",
    notes="Pushing enemy base; enemy has halberdiers defending",
)

prompt3 = _snapshot_to_prompt(imperial_snap, deque(), PLAYER_CIV)
result3 = supervisor.invoke({"messages": [{"role": "user", "content": prompt3}]})
print(_extract_last_content(result3))

## 5 — Verify Delegation: Which Sub-agents Did the Supervisor Call?

Introspect the message chain to confirm the supervisor actually delegated to
sub-agent tools rather than answering from memory.

In [ ]:
from collections import Counter

def tool_call_summary(result: dict) -> dict:
    """Count how many times each sub-agent tool was called."""
    counts = Counter()
    for msg in result.get("messages", []):
        for tc in getattr(msg, "tool_calls", []):
            counts[tc["name"]] += 1
    return dict(counts)

print("Dark Age scenario tool calls:    ", tool_call_summary(result))
print("Castle attack scenario tool calls:", tool_call_summary(result2))
print("Imperial siege scenario tool calls:", tool_call_summary(result3))

## 6 — Streaming Output from the Supervisor

In [ ]:
print("=== Streaming strategy update (Castle attack scenario) ===")

for chunk in supervisor.stream(
    {"messages": [{"role": "user", "content": prompt2}]},
    stream_mode="values",
):
    last_msg = chunk["messages"][-1]
    last_msg.pretty_print()

## 7 — Async Single Tick (mirrors StrategistLoop)

In [ ]:
import asyncio

async def one_tick(snap: GameStateSnapshot, player_civ: str, opponent_civ: str) -> str:
    """Simulate one StrategistLoop tick asynchronously."""
    prompt_text = _snapshot_to_prompt(snap, deque(), player_civ, opponent_civ)
    result = await asyncio.to_thread(
        supervisor.invoke,
        {"messages": [{"role": "user", "content": prompt_text}]},
    )
    return _extract_last_content(result)

update = await one_tick(dark_snap, PLAYER_CIV, OPPONENT_CIV)
print("=== Async tick result ===")
print(update)

## 8 — Different Civilization: Britons (Archer Civ)

In [ ]:
# Build a Britons vs Mongols strategist — matchup analyst is seeded with both
BRITONS_OPPONENT = "Mongols"
britons_supervisor = build_strategist("Britons", BRITONS_OPPONENT)

feudal_snap = GameStateSnapshot(
    age="Feudal",
    population=22,
    pop_cap=30,
    resources={"food": 350, "wood": 400, "gold": 80, "stone": 0},
    visible_units=[
        DetectedUnit(label="villager", owner="self", approx_count=18),
        DetectedUnit(label="archer", owner="self", approx_count=4),
        DetectedUnit(label="scout cavalry", owner="enemy", approx_count=3),
    ],
    visible_buildings=["Town Center", "Archery Range", "Barracks", "Mill"],
    idle_villagers_visible=False,
    threat_level="low",
    notes="Enemy scouts pressuring the economy",
)

prompt_britons = _snapshot_to_prompt(feudal_snap, deque(), "Britons", BRITONS_OPPONENT)
result_britons = britons_supervisor.invoke({"messages": [{"role": "user", "content": prompt_britons}]})
print(_extract_last_content(result_britons))